# 08 Hypocentres

How well does each permanent network (GeoNet stations, plus Trilliums) locate earthquakes? This matters for the
future catalogue, not for matrix imaging. Builds a synthetic catalogue (regional events resampled from the relocated
catalogue, plus induced events after Sherburn et al. 2015), a detection model fitted to GeoNet picks, 3D P and S
travel times, synthetic picks, and locations with a 1D model.

Inputs: `data/site.json`, `data/velocity_model.npz`, `data/stations.csv`, `data/designs.json`,
`data/relocated_locations.csv`, `data/TVZ_*.xml`. Outputs in `data/` (not shared).

In [ ]:
# --- Imports ---
import json, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfmm
from obspy import read_events
from obspy.clients.fdsn import Client
from scipy.interpolate import RectBivariateSpline
from scipy.ndimage import map_coordinates
from scipy.optimize import least_squares, minimize

# --- Site (private) ---
site = json.load(open("../data/site.json"))
DRILL = (site["drill_site"]["lon"], site["drill_site"]["lat"])
ELEV = site["elevation_km"]
KY = 111.32
KX = KY * np.cos(np.radians(DRILL[1]))
km = lambda lon, lat: ((np.asarray(lon) - DRILL[0]) * KX, (np.asarray(lat) - DRILL[1]) * KY)

In [ ]:
rng = np.random.default_rng(1)

# --- Networks ---
GEONET_R = 13.0                        # GeoNet stations within this distance (km); must fit inside the model grid

# --- Detection model (fitted to GeoNet picks) ---
QML_YEARS = range(2019, 2024)          # QuakeML files used for the fit
R_MAX = 60.0                           # event-station pairs used for the fit (km)
MIN_PICKS = 10                         # stations with fewer P picks are left out of the fit
UPTIME = 0.95                          # best-case pick rate for our own stations (replaces the fitted ceiling). assumed

# --- Synthetic catalogue ---
YEARS = 3                              # deployment length
CROP = 13.0                            # events within +/- 13 km
M_MIN = 1.0                            # regional events at or above this magnitude
CAT_YEARS = 10.0                       # time span of relocated_locations.csv (years). check
CAT_COLS = dict(lon="lon", lat="lat", depth="depth", mag="mag")   # column names in the file. check
INJ_CENTRE, INJ_R = (1.5, 0.5), 1.0    # injection zone, km from drill site. placeholder
INJ_RATE, INJ_B = 222, 0.96            # induced events per year, b-value (Sherburn et al. 2015)
INJ_M = (0.8, 3.1)                     # induced magnitude range (Sherburn et al. 2015)
INJ_Z = (1.5, 3.0)                     # induced depths, km bsl (Sherburn et al. 2015)

# --- Picks and locating ---
SIGMA = {"p": 0.2, "s": 0.4}         # pick errors (s), assumed
MIN_P = 4                              # fewest P picks needed to locate
R0 = 0.3                               # starting sphere for travel times (km)

In [ ]:
# --- GeoNet stations near Rotokawa ---
inv = Client("GEONET").get_stations(network="NZ", latitude=DRILL[1], longitude=DRILL[0],
                                    maxradius=0.6, level="station")
gn = pd.DataFrame([(s.code, s.longitude, s.latitude, s.start_date, s.end_date)
                   for net in inv for s in net], columns=["code", "lon", "lat", "start", "end"])
gn["x"], gn["y"] = km(gn.lon, gn.lat)

# currently operating, within GEONET_R
near = gn[(np.hypot(gn.x, gn.y) <= GEONET_R) & gn.end.isna()]
print("GeoNet stations used:", sorted(near.code))

# --- Our Trilliums (from 02) ---
st02 = pd.read_csv("../data/stations.csv", index_col="station")
trill = json.load(open("../data/designs.json"))["trillium"]

net_st = pd.concat([pd.DataFrame(dict(code=near.code, x=near.x, y=near.y, kind="geonet")),
                    pd.DataFrame(dict(code=[f"T{k:02d}" for k in range(len(trill))],
                                      x=st02.x[trill].values, y=st02.y[trill].values, kind="trillium"))],
                   ignore_index=True)

NETWORKS = {"GeoNet only": net_st.index[net_st.kind == "geonet"].tolist(),
            "GeoNet + Trilliums": net_st.index.tolist()}

In [ ]:
PAIRS = "../data/det_pairs.csv"
if not os.path.exists(PAIRS):
    rows = []
    for yr in QML_YEARS:
        for ev in read_events(f"../data/TVZ_{yr}.xml"):
            o = ev.preferred_origin() or ev.origins[0]
            mg = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
            if mg is None or o.depth is None:
                continue
            picked = {(p.waveform_id.station_code, p.phase_hint[0].lower())
                      for p in ev.picks if p.phase_hint and p.phase_hint[0] in "PSps"}
            ex, ey = km(o.longitude, o.latitude)
            on = gn[(gn.start <= o.time) & (gn.end.isna() | (gn.end >= o.time))]       # stations running then
            R = np.sqrt((on.x - ex) ** 2 + (on.y - ey) ** 2 + (o.depth / 1000) ** 2)
            for code, r in zip(on.code[R <= R_MAX], R[R <= R_MAX]):
                rows.append((mg.mag, r, code, (code, "p") in picked, (code, "s") in picked))
    pd.DataFrame(rows, columns=["mag", "R", "code", "p", "s"]).to_csv(PAIRS, index=False)

pairs = pd.read_csv(PAIRS)
good = pairs.groupby("code").p.sum()
pairs = pairs[pairs.code.isin(good.index[good >= MIN_PICKS])]                         # stations that actually pick


def nll(c, M, logR, det):
    """Negative log-likelihood of the detection model."""
    p = np.clip(np.clip(c[3], 0.01, 1) / (1 + np.exp(-(c[0] + c[1] * M + c[2] * logR))), 1e-9, 1 - 1e-9)
    return -np.sum(det * np.log(p) + (1 - det) * np.log(1 - p))


coef = {}
for ph in ("p", "s"):
    f = minimize(nll, [0, 2, -3, 0.9], args=(pairs.mag.values, np.log10(pairs.R.values), pairs[ph].values),
                 method="Nelder-Mead", options=dict(maxiter=5000))
    coef[ph] = f.x
    print(ph, "c0..c3 =", np.round(f.x, 2))

In [ ]:
# --- Regional: real relocated events in the area, resampled to the deployment length ---
cat = pd.read_csv("../data/relocated_locations.csv").rename(columns={v: k for k, v in CAT_COLS.items()})
cat["x"], cat["y"] = km(cat.lon, cat.lat)
cat = cat[(cat.x.abs() <= CROP) & (cat.y.abs() <= CROP) & (cat.mag >= M_MIN)]
n_reg = rng.poisson(len(cat) * YEARS / CAT_YEARS)
reg = cat.sample(n_reg, replace=True, random_state=1)[["x", "y", "depth", "mag"]].assign(source="regional")
# depths are assumed below sea level; convert to below ground. check the catalogue's convention
reg["depth"] += ELEV

# --- Induced: Sherburn et al. 2015 rate, depths and b-value, in the injection zone ---
n_ind = rng.poisson(INJ_RATE * YEARS)
r, a = INJ_R * np.sqrt(rng.uniform(size=n_ind)), rng.uniform(0, 2 * np.pi, n_ind)
u = rng.uniform(size=n_ind)                                                         # Gutenberg-Richter, truncated
mag = INJ_M[0] - np.log10(1 - u * (1 - 10 ** (-INJ_B * (INJ_M[1] - INJ_M[0])))) / INJ_B
ind = pd.DataFrame(dict(x=INJ_CENTRE[0] + r * np.sin(a), y=INJ_CENTRE[1] + r * np.cos(a),
                        depth=rng.uniform(*INJ_Z, n_ind) + ELEV, mag=mag, source="induced"))

events = pd.concat([reg, ind], ignore_index=True)
events.to_csv("../data/synthetic_catalogue.csv", index_label="event")
print(events.source.value_counts().to_dict())

In [ ]:
m = np.load("../data/velocity_model.npz")
x, y, z = m["x"], m["y"], m["z"]
vel = {"p": m["vp"], "s": m["vs"]}
dx = [x[1] - x[0], y[1] - y[0], z[1] - z[0]]
XX, YY, ZZ = np.meshgrid(x, y, z, indexing="ij")
TT = "../data/tt_hypo"
os.makedirs(TT, exist_ok=True)


def travel_time(sx, sy, v):
    """Travel time (s) from a surface station to every cell."""
    d = np.sqrt((XX - sx) ** 2 + (YY - sy) ** 2 + ZZ ** 2)
    v0 = v[np.argmin(abs(x - sx)), np.argmin(abs(y - sy)), 0]
    t = skfmm.travel_time(d - R0, v, dx=dx, order=2)
    return np.where(d < R0, d / v0, t + R0 / v0).astype(np.float16)


for _, s in net_st.iterrows():
    for ph in ("p", "s"):
        path = f"{TT}/{s.code}_{ph}.npy"
        if not os.path.exists(path):
            np.save(path, travel_time(s.x, s.y, vel[ph]))

# 1D locating model: 3D model averaged at each depth
r1 = np.arange(0, 35.01, 0.1)
d1 = np.hypot(*np.meshgrid(r1, z, indexing="ij"))
tt1d = {}
for ph in ("p", "s"):
    v1 = vel[ph].mean(axis=(0, 1))
    t = skfmm.travel_time(d1 - R0, np.tile(v1, (len(r1), 1)), dx=[0.1, dx[2]], order=2)
    tt1d[ph] = RectBivariateSpline(r1, z, np.where(d1 < R0, d1 / v1[0], t + R0 / v1[0]))

In [ ]:
ev = events[["x", "y", "depth"]].values
idx = ((ev - [x[0], y[0], z[0]]) / dx).T
true = {ph: np.column_stack([map_coordinates(np.load(f"{TT}/{c}_{ph}.npy").astype(np.float32), idx, order=1)
                             for c in net_st.code]) for ph in ("p", "s")}
picks = {ph: true[ph] + rng.normal(0, SIGMA[ph], true[ph].shape) for ph in true}

R = np.sqrt((ev[:, None, 0] - net_st.x.values) ** 2 + (ev[:, None, 1] - net_st.y.values) ** 2 + ev[:, None, 2] ** 2)
M = events.mag.values[:, None]
u = rng.uniform(size=R.shape)
detected = {ph: u < UPTIME / (1 + np.exp(-(c[0] + c[1] * M + c[2] * np.log10(R)))) for ph, c in coef.items()}

In [ ]:
sx, sy = net_st.x.values, net_st.y.values


def locate(ids, e):
    """Locate event e with stations ids and the 1D model. Returns (x, y, z) or None."""
    ip = [s for s in ids if detected["p"][e, s]]
    js = [s for s in ids if detected["s"][e, s]]
    if len(ip) < MIN_P:
        return None
    obs = np.r_[picks["p"][e, ip], picks["s"][e, js]]
    w = np.r_[np.full(len(ip), 1 / SIGMA["p"]), np.full(len(js), 1 / SIGMA["s"])]

    def resid(q):
        rp, rs = np.hypot(sx[ip] - q[0], sy[ip] - q[1]), np.hypot(sx[js] - q[0], sy[js] - q[1])
        return (obs - q[3] - np.r_[tt1d["p"].ev(rp, q[2]), tt1d["s"].ev(rs, q[2])]) * w

    first = ip[np.argmin(picks["p"][e, ip])]
    best = None
    for z0 in (2.0, 5.0, 10.0):
        t0 = picks["p"][e, first] - tt1d["p"].ev(0, z0)
        fit = least_squares(resid, [sx[first], sy[first], z0, t0],
                            bounds=([-np.inf, -np.inf, 0, -np.inf], [np.inf, np.inf, z[-1], np.inf]))
        best = fit if best is None or fit.cost < best.cost else best
    return best.x[:3]


rows = []
for name, ids in NETWORKS.items():
    for e in range(len(events)):
        loc = locate(ids, e)
        err = np.full(3, np.nan) if loc is None else loc - ev[e]
        rows.append((name, events.source[e], ev[e, 2], *err))
res = pd.DataFrame(rows, columns=["network", "source", "z_true", "ex", "ey", "ez"])
res["eh"] = np.hypot(res.ex, res.ey)

In [ ]:
summary = res.groupby(["network", "source"]).agg(
    located=("ez", lambda s: s.notna().mean()),
    median_eh=("eh", "median"),
    median_ez=("ez", lambda s: s.abs().median()),
    bias_ez=("ez", "median")).round(2)
print(summary)

In [ ]:
bins = np.arange(0, 21, 1)
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(6, 4))
    for name, grp in res.dropna().groupby("network"):
        b = pd.cut(grp.z_true, bins)
        stat = grp.groupby(b, observed=True).ez.agg(["median", "count"])
        stat = stat[stat["count"] >= 20]
        ax.plot([iv.mid for iv in stat.index], stat["median"], "o-", label=name)
    ax.axhline(0, color="k", lw=0.8)
    ax.set(xlabel="True depth (km)", ylabel="Median depth error (km)", title="Depth bias (bins with 20+ events)")
    ax.legend()
    plt.show()